In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import kendalltau, pearsonr, spearmanr


## Settings

In [ ]:
# Set display options for pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
# Define dataset variable configurations
DATASET_VARS = {
    "BH_1": {
        "obj_var": "yield",
        "cat_vars": [
            "Aryl_halide_SMILES",
            "Additive_SMILES",
            "Base_SMILES",
            "Ligand_SMILES",
        ],
        "con_vars": [],
    },
    "DA": {
        "obj_var": "yield",
        "cat_vars": ["Base_SMILES", "Ligand_SMILES", "Solvent_SMILES"],
        "con_vars": ["Concentration", "Temp_C"],
    },
    "alkox": {
        "obj_var": "conversion",
        "cat_vars": [],
        "con_vars": ["catalase", "peroxidase", "alcohol_oxidase", "ph"],
    },
    "oer_plate_a": {
        "obj_var": "overpotential",
        "cat_vars": [],
        "con_vars": ["ni_load", "fe_load", "co_load", "mn_load", "ce_load", "la_load"],
    },
    "p3ht": {
        "obj_var": "conductivity",
        "cat_vars": [],
        "con_vars": [
            "p3ht_content",
            "d1_content",
            "d2_content",
            "d6_content",
            "d8_content",
        ],
    },
    "photo_pce10": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "photo_wf3": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "suzuki_edbo": {
        "obj_var": "yield",
        "cat_vars": ["electrophile", "nucleophile", "base", "ligand", "solvent"],
        "con_vars": [],
    },
    "suzuki": {
        "obj_var": "yield",
        "cat_vars": [],
        "con_vars": ["temperature", "pd_mol", "arbpin", "k3po4"],
    },
}

In [ ]:
DATASET_NAME_ORDER = [
    "BH_1",
    "DA",
    "alkox",
    "oer_plate_a",
    "p3ht",
    "photo_pce10",
    "photo_wf3",
    "suzuki_edbo",
    "suzuki",
]
MODEL_ORDER = [
    "gpt-5-mini-2025-08-07",
    "o4-mini-2025-04-16",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4o-mini-2024-07-18",
    "claude-sonnet-4-5-20250929",
    "claude-haiku-4-5-20251001",
    "claude-3-5-haiku-20241022",
]
MODEL_LABELS = {
    "gpt-5-mini-2025-08-07": "GPT-5 mini",
    "o4-mini-2025-04-16": "o4 mini",
    "gpt-4.1-mini-2025-04-14": "GPT-4.1 mini",
    "gpt-4o-mini-2024-07-18": "GPT-4o mini",
    "claude-sonnet-4-5-20250929": "Claude Sonnet 4.5",
    "claude-haiku-4-5-20251001": "Claude Haiku 4.5",
    "claude-3-5-haiku-20241022": "Claude 3.5 Haiku",
}

In [ ]:
TIME_TAGS = [
    "20251105232125",
    "20251105232200",
    "20251105232320",
    "20251106221207",
    "20251106221240",
    "20251106221304",
]

In [ ]:
# Read and concatenate batch output logs
batch_output_logs_df_list = []
for time_tag in TIME_TAGS:
    batch_output_logs_df_tmp = pd.read_csv(
        Path("./results_final") / f"results_{time_tag}" / "batch_output_logs.csv"
    )
    batch_output_logs_df_tmp.insert(2, "time_tag", time_tag)
    batch_output_logs_df_list.append(batch_output_logs_df_tmp)
batch_output_logs_df = pd.concat(batch_output_logs_df_list, axis=0, ignore_index=True)

# Sort by dataset_name and model
batch_output_logs_df["dataset_name"] = pd.Categorical(
    batch_output_logs_df["dataset_name"], categories=DATASET_NAME_ORDER, ordered=True
)
batch_output_logs_df["model"] = pd.Categorical(
    batch_output_logs_df["model"], categories=MODEL_ORDER, ordered=True
)
batch_output_logs_df = batch_output_logs_df.sort_values(
    ["dataset_name", "model"]
).reset_index(drop=True)

# Convert to string type
batch_output_logs_df["dataset_name"] = batch_output_logs_df["dataset_name"].astype(str)
batch_output_logs_df["model"] = batch_output_logs_df["model"].astype(str)

# Create a new column containing model parameters
batch_output_logs_df["model_w_params"] = batch_output_logs_df["model"].map(MODEL_LABELS)
for idx, row in batch_output_logs_df.iterrows():
    if pd.notna(batch_output_logs_df.loc[idx, "reasoning_effort"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += (
            " " + batch_output_logs_df.loc[idx, "reasoning_effort"]
        )
    elif pd.notna(batch_output_logs_df.loc[idx, "temperature"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += " temp" + str(
            round(batch_output_logs_df.loc[idx, "temperature"], 2)
        )

# Save the summarized batch output logs
batch_output_logs_df.to_csv("results_final/batch_output_logs.csv", index=False)

## Scores

In [ ]:
# Scores
scores = []
exp_result_dfs = {}

for i, batch_output_logs_df_row in batch_output_logs_df[
    ["dataset_name", "model", "model_w_params", "time_tag"]
].iterrows():
    dataset_name = batch_output_logs_df_row.dataset_name
    model = batch_output_logs_df_row.model
    model_w_params = batch_output_logs_df_row.model_w_params
    time_tag = batch_output_logs_df_row.time_tag

    print(f"dataset: {dataset_name}, model {model_w_params}")

    # Read experimental data
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    exp_df = pd.read_csv(f"./dataset_processed/dataset_{dataset_name}_extracted.csv")
    exp_df = exp_df.rename(columns={obj_var: f"{obj_var}_true"})

    # Read prediction results
    result_df = pd.read_csv(
        Path("./results_final")
        / f"results_{time_tag}"
        / f"dataset_{dataset_name}_{model}_result.csv"
    )
    result_df = result_df.rename(columns={"predicted_value": f"{obj_var}_pred"})

    # Merge results
    result_df = pd.merge(result_df, exp_df, on="ID", how="left")
    exp_result_dfs[(dataset_name, model_w_params, time_tag)] = result_df

    # Calculate correlation coefficients
    score_dict = {
        "dataset_name": dataset_name,
        "model": model,
        "model_w_params": model_w_params,
        "time_tag": time_tag,
    }

    y_true = result_df[f"{obj_var}_true"].to_numpy()
    y_pred = result_df[f"{obj_var}_pred"].to_numpy()
    pearson_corr = pearsonr(y_true, y_pred).statistic
    pearson_corr_pvalue = pearsonr(y_true, y_pred).pvalue
    spearman_corr = spearmanr(y_true, y_pred).statistic
    spearman_corr_pvalue = spearmanr(y_true, y_pred).pvalue
    kt_corr = kendalltau(y_true, y_pred).statistic
    kt_corr_pvalue = kendalltau(y_true, y_pred).pvalue

    score_dict.update(
        {
            "pearson_corr": pearson_corr,
            "pearson_corr_pvalue": pearson_corr_pvalue,
            "spearman_corr": spearman_corr,
            "spearman_corr_pvalue": spearman_corr_pvalue,
            "kt_corr": kt_corr,
            "kt_corr_pvalue": kt_corr_pvalue,
        }
    )

    scores.append(score_dict)

scores_df = pd.DataFrame(scores)

# Save scores
scores_df.to_csv("results_final/scores.csv", index=False)

In [ ]:
# Plot Pearson's correlation coefficient bar chart
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=scores_df, x="dataset_name", y="pearson_corr", hue="model_w_params"
)
ax.axhline(0, color="black", linewidth=0.5)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.ylim(-0.4, 1.0)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(np.arange(-0.4, 1.1, 0.1), fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Pearson's \ncorrelation coefficient", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.savefig("images/direct_pearson.png", format="png", dpi=600, bbox_inches="tight")
plt.savefig("images/direct_pearson.pdf", format="pdf", bbox_inches="tight")
plt.savefig("images/direct_pearson.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# Plot Spearman's rank correlation coefficient bar chart
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=scores_df, x="dataset_name", y="spearman_corr", hue="model_w_params"
)
ax.axhline(0, color="black", linewidth=0.5)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.ylim(-0.4, 1.0)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(np.arange(-0.4, 1.1, 0.1), fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Spearman's rank \ncorrelation coefficient", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.savefig("images/direct_spearman.png", format="png", dpi=600, bbox_inches="tight")
plt.savefig("images/direct_spearman.pdf", format="pdf", bbox_inches="tight")
plt.savefig("images/direct_spearman.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# Plot Kendall's rank correlation coefficient bar chart
plt.figure(figsize=(6, 4))
ax = sns.barplot(data=scores_df, x="dataset_name", y="kt_corr", hue="model_w_params")
ax.axhline(0, color="black", linewidth=0.5)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.ylim(-0.4, 1.0)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(np.arange(-0.4, 1.1, 0.1), fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Kendall's rank \ncorrelation coefficient", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.savefig("images/direct_kendall.png", format="png", dpi=600, bbox_inches="tight")
plt.savefig("images/direct_kendall.pdf", format="pdf", bbox_inches="tight")
plt.savefig("images/direct_kendall.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# Scatter plots of predicted vs actual values
ncol = len(np.unique([model_w_params for _, model_w_params, _ in exp_result_dfs]))
nsub = len(exp_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 3, nrow * 3))
for i, ((dataset_name, model_w_params, _), exp_result_df) in enumerate(
    exp_result_dfs.items()
):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]

    plt.subplot(nrow, ncol, i + 1)
    sns.scatterplot(
        data=exp_result_df,
        x=f"{obj_var}_true",
        y=f"{obj_var}_pred",
        s=20,
        alpha=0.5,
        color="red",
    )
    y_lim_min = np.min(
        [exp_result_df[f"{obj_var}_true"].min(), exp_result_df[f"{obj_var}_pred"].min()]
    )
    y_lim_max = np.max(
        [exp_result_df[f"{obj_var}_true"].max(), exp_result_df[f"{obj_var}_pred"].max()]
    )
    y_lim_min, y_lim_max = (
        y_lim_min - (y_lim_max - y_lim_min) * 0.05,
        y_lim_max + (y_lim_max - y_lim_min) * 0.05,
    )
    plt.plot(
        [y_lim_min, y_lim_max],
        [y_lim_min, y_lim_max],
        linestyle="--",
        color="black",
        alpha=0.5,
    )
    plt.xlim(y_lim_min, y_lim_max)
    plt.ylim(y_lim_min, y_lim_max)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"Actual {obj_var}", fontsize=10)
    plt.ylabel(f"Predicted {obj_var}", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig("images/direct_scatter.png", format="png", dpi=300, bbox_inches="tight")
plt.savefig("images/direct_scatter.pdf", format="pdf", bbox_inches="tight")
plt.savefig("images/direct_scatter.svg", format="svg", bbox_inches="tight")
plt.show()